In [ ]:
import pandas as pd

In [2]:
cars= pd.read_csv(r"C:\Users\acreddy\Desktop\abc\DecisionTree_Classfier_and_Reg\data\cars_preprocessed_data.csv")

In [3]:
cars.head(2)

,symboling,wheel_base,length,width,height,curb_weight,num_of_cylinders,engine_size,bore,stroke,compression_ratio,horsepower,peak_rpm,city_mpg,highway_mpg,price
0,3,88.6,168.8,64.1,48.8,2548,4,130,3.47,2.68,9.0,111.0,5000.0,21,27,13495.0
1,3,88.6,168.8,64.1,48.8,2548,4,130,3.47,2.68,9.0,111.0,5000.0,21,27,16500.0


In [5]:
# divide data into independent vars and target
X= cars.iloc[:,:-1]
y= cars.iloc[:,-1]

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
# split the data in to train and test sets
X_train, X_test, y_train, y_test= train_test_split(X,y, test_size=0.25, random_state=42)

In [9]:
len(X_train), len(y_train)

(153, 153)

In [10]:
len(X_test), len(y_test)

(52, 52)

In [11]:
from sklearn.preprocessing import StandardScaler

In [12]:
# initialize the standardscaler
scaler= StandardScaler()

In [14]:
# fit_trainsform on train data and transform on test data
X_train_scaled= scaler.fit_transform(X_train)
X_test_scaled= scaler.transform(X_test)

# Hyper Param Tuning Using Combination of Random and GridSearchCV

In [15]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.tree import DecisionTreeRegressor

In [16]:
# initialize regressor
regressor= DecisionTreeRegressor()

In [17]:
# defining decision tree classifier params in RandomSearchCV as param_dist_random
param_dist_random= {
    "max_depth": list(np.arange(3,7, 1)), # this will start with 3,4,5,6,7 as max depth of tree
    "min_samples_split": list(np.arange(2, 11, 1)), # the min samples to split the node. it will try 2,3,..11 
    "min_samples_leaf": list(np.arange(1, 5, 1))   # the min samples required to be a leaf. it will try 1,2,3..5
}

In [18]:
# Use RandomizedSearchCV to perform a random search
random_search = RandomizedSearchCV(
    regressor, # i have defined the above i,e labelEncoding and DecisionTreeClassifier together pipeline
    param_distributions=param_dist_random, 
    n_iter=10, 
    cv=5, 
    scoring="r2", 
    random_state=42
)

In [19]:
# Fit the random_search model to the train data
random_search.fit(X_train_scaled, y_train)

RandomizedSearchCV(cv=5, estimator=DecisionTreeRegressor(),
                   param_distributions={'max_depth': [3, 4, 5, 6],
                                        'min_samples_leaf': [1, 2, 3, 4],
                                        'min_samples_split': [2, 3, 4, 5, 6, 7,
                                                              8, 9, 10]},
                   random_state=42, scoring='r2')

In [20]:
best_params_random= random_search.best_params_

In [21]:
best_params_random

{'min_samples_split': 8, 'min_samples_leaf': 3, 'max_depth': 6}

In [22]:
# Refine the hyperparameter grid for GridSearchCV based on the results from random search
refined_param_grid = {
    "max_depth":list(np.arange(best_params_random["max_depth"] - 2, 
                                        best_params_random["min_samples_split"] + 3, 1)
                    ),
    "min_samples_split": list(np.arange(best_params_random["min_samples_split"] - 2, 
                                        best_params_random["min_samples_split"] + 3, 1)
                             ),
    "min_samples_leaf": list(np.arange(best_params_random["min_samples_leaf"] - 1, 
                                       best_params_random["min_samples_leaf"] + 2, 1)
                            )
}

In [23]:
# Perform Grid Search around the best parameters from Randomized Search
random_grid_search = GridSearchCV(
        regressor, 
        param_grid= refined_param_grid,
        cv=5,       # Cross-validation folds
        scoring="r2",
        verbose=1
    )

In [24]:
# refined params gridsearch fit on train data
random_grid_search.fit(X_train_scaled, y_train)

Fitting 5 folds for each of 105 candidates, totalling 525 fits


GridSearchCV(cv=5, estimator=DecisionTreeRegressor(),
             param_grid={'max_depth': [4, 5, 6, 7, 8, 9, 10],
                         'min_samples_leaf': [2, 3, 4],
                         'min_samples_split': [6, 7, 8, 9, 10]},
             scoring='r2', verbose=1)

In [25]:
# Get the best hyperparameters from Grid Search
best_params_grid = random_grid_search.best_params_

In [26]:
best_params_grid

{'max_depth': 7, 'min_samples_leaf': 2, 'min_samples_split': 10}

In [28]:
# Compare the results
print("Best Hyperparameters from Randomized Search:", best_params_random)
print("Best Hyperparameters from Grid Search:", best_params_grid)
print("Best R2 Score: ", random_grid_search.best_score_)

Best Hyperparameters from Randomized Search: {'min_samples_split': 8, 'min_samples_leaf': 3, 'max_depth': 6}
Best Hyperparameters from Grid Search: {'max_depth': 7, 'min_samples_leaf': 2, 'min_samples_split': 10}
Best R2 Score:  0.8340802838961384


In [29]:
# Evaluate on the test set
best_model_grid = random_grid_search.best_estimator_
y_pred= best_model_grid.predict(X_test_scaled)

In [30]:
# lets get the metrics
from sklearn.metrics import r2_score, mean_squared_error

In [31]:
r2= r2_score(y_test, y_pred)
mse= mean_squared_error(y_test, y_pred)

In [33]:
r2

0.8609282280650249

In [34]:
mse

9339288.747501003

Note: Decision Tree Regressor did a fantastic job than ElasticNet and LinearRegression. this is the max R2 score among these 3 models.